# 01 — Just watch

This notebook runs the equation and shows you what happens. No physics background required. No code edits required.

**What you will see:** starting from a smooth Gaussian-shaped initial state in three dimensions, the equation produces a body-centered cubic (BCC) crystalline pattern that emerges spontaneously from the dynamics. The crystal has a definite spacing and a definite symmetry that you can read off from the Fourier spectrum at the end.

**Run time:** approximately 1–3 minutes on a Colab GPU, longer on CPU.

**To run:** click the play button on each cell in order, or use *Runtime → Run all*.

## Setup

Install the required packages. On Colab this takes about a minute. Local users with CuPy already installed can skip this cell.

In [ ]:
import sys
if 'google.colab' in sys.modules:
    !pip install -q cupy-cuda12x numpy matplotlib

In [ ]:
import numpy as np
import matplotlib.pyplot as plt

try:
    import cupy as xp
    print('Using GPU via CuPy:', xp.cuda.runtime.getDeviceProperties(0)['name'].decode())
    on_gpu = True
except ImportError:
    import numpy as xp
    print('CuPy not available; falling back to NumPy on CPU.')
    on_gpu = False

## The equation in three dimensions

We integrate the memory-augmented nonlinear Schrödinger equation on a 96³ lattice with periodic boundary conditions. The parameters are chosen to place the system in the released crystalline window:

- **Lattice:** $96^3$ voxels
- **Box length:** $L = 20$
- **Time step:** $dt = 0.0025$
- **Nonlinearity:** $\Lambda = -8$
- **Memory:** two-mode, total $\Sigma\lambda = 1.5$ (fast $\nu_1 = 10$, slow $\nu_2 = 0.5$)
- **Initial state:** Gaussian, $\sigma_0 = 0.5$, centered, no initial momentum
- **Warmup:** 1500 steps (system passes through collapse and release)
- **Display:** snapshots at end of warmup

The total runtime is roughly $1500 \times 96^3$ field updates; this is fast on GPU and slow on CPU.

In [ ]:
# Lattice and physical parameters
N = 96
L = 20.0
dt = 0.0025
n_steps = 1500
Lambda = -8.0
sigma_0 = 0.5

# Memory: two modes
nus = [10.0, 0.5]
lambdas = [1.125, 0.375]   # total Sigma_lambda = 1.5, split 3:1 fast:slow

# Build the initial Gaussian state
dx = L / N
x = (xp.arange(N) - N/2) * dx
X, Y, Z = xp.meshgrid(x, x, x, indexing='ij')
r2 = X**2 + Y**2 + Z**2
psi = xp.exp(-r2 / (2 * sigma_0**2)).astype(xp.complex64)
norm = xp.sqrt(xp.sum(xp.abs(psi)**2) * dx**3)
psi /= norm
print(f'Initial peak density: {float(xp.max(xp.abs(psi)**2)):.4f}')

# Build the kinetic propagator in k-space
kvec = xp.fft.fftfreq(N, d=dx) * 2 * np.pi
kx, ky, kz = xp.meshgrid(kvec, kvec, kvec, indexing='ij')
k2 = kx**2 + ky**2 + kz**2
H_kin = 0.5 * k2
U_k = xp.exp(-1j * H_kin * dt).astype(xp.complex64)

# Initialize auxiliary memory fields
ys = [xp.zeros_like(psi, dtype=xp.float32) for _ in nus]
decays = [float(np.exp(-nu * dt)) for nu in nus]

## Run the integration

Each time step is a Strang split: half potential step, full kinetic step, half potential step, then exact OU update of the auxiliary memory fields. We print the peak density every 100 steps so you can watch the dynamics: an initial collapse spike, then the memory releases the field outward.

In [ ]:
import time
t0 = time.time()

for step in range(n_steps):
    # Compute density and memory potential
    rho = (psi.conj() * psi).real.astype(xp.float32)
    V_mem = sum(lam * y for lam, y in zip(lambdas, ys))
    V_tot = Lambda * rho + V_mem
    
    # Half potential step
    psi = psi * xp.exp(-1j * V_tot.astype(xp.complex64) * dt * 0.5)
    
    # Full kinetic step in k-space
    psi = xp.fft.ifftn(U_k * xp.fft.fftn(psi))
    
    # Half potential step with updated density
    rho = (psi.conj() * psi).real.astype(xp.float32)
    V_tot = Lambda * rho + V_mem
    psi = psi * xp.exp(-1j * V_tot.astype(xp.complex64) * dt * 0.5)
    
    # OU update of auxiliary fields
    for i, (decay, nu) in enumerate(zip(decays, nus)):
        ys[i] = decay * ys[i] + (1 - decay) * rho
    
    if step % 100 == 0:
        peak = float(xp.max(rho))
        print(f'step {step:4d}  t = {step*dt:6.3f}  peak density = {peak:.4f}')

elapsed = time.time() - t0
print(f'\nDone. Wall time: {elapsed:.1f}s')

## Visualize the result

We look at the final density field. The crystalline structure is more easily seen in Fourier space (the bright spots in the power spectrum directly reveal the lattice symmetry) than in real space. We plot both.

In [ ]:
rho_final = (psi.conj() * psi).real
rho_np = rho_final.get() if on_gpu else rho_final

# Power spectrum
psi_k = xp.fft.fftshift(xp.fft.fftn(psi))
P = (psi_k.conj() * psi_k).real
P_np = P.get() if on_gpu else P

# Take an equatorial slice for visualization
rho_slice = rho_np[:, :, N//2]
P_slice = P_np[:, :, N//2]

fig, axes = plt.subplots(1, 2, figsize=(12, 5))
axes[0].imshow(rho_slice, cmap='viridis')
axes[0].set_title('Final density (real-space slice)')
axes[0].set_xlabel('x'); axes[0].set_ylabel('y')
im = axes[1].imshow(np.log10(P_slice + 1e-20), cmap='magma')
axes[1].set_title('log10 power spectrum (k-space slice)')
axes[1].set_xlabel('kx'); axes[1].set_ylabel('ky')
plt.tight_layout()
plt.show()

print(f'\nFinal peak density: {float(xp.max(rho_final)):.4f}')
print(f'Initial peak density was: 1.4377')
print(f'Ratio: peak has been reduced by approximately {1.4377/float(xp.max(rho_final)):.1f}x')

## What you should see

The real-space slice shows the field is no longer a single Gaussian. It has spread out across the box and developed structure.

The power spectrum slice shows a roughly hexagonal pattern of bright spots at moderate radial wavenumber. These bright spots are the signature of the body-centered cubic (BCC) crystal structure projected onto the equatorial plane of $k$-space. A full three-dimensional analysis of the BCC selection is in [`../implementation/physics/observables.py`](../implementation/physics/observables.py) and is run by [`../experiments/physics/reproduce_3d_bravais_sweep.py`](../experiments/physics/reproduce_3d_bravais_sweep.py).

The peak density has dropped substantially from its initial value. This is the anti-collapse mechanism in action: without the memory potential, the field would have collapsed to lattice scale and remained locked there; with the memory potential, the field has released and now occupies an extended crystalline configuration.

## What to explore next

- [`02-adjust-the-knobs.ipynb`](02-adjust-the-knobs.ipynb): interactive sliders for the equation parameters.
- [`../results/05-bravais-selection.md`](../results/05-bravais-selection.md): the full Bravais selection result.
- [`../results/06-dimensional-rescaling.md`](../results/06-dimensional-rescaling.md): why the memory coupling has to be $\sim 1.5$ rather than $\sim 0.15$ in 3D.